# Forecast a cafe's daily sales

A campus cafe has logged the cups of coffee it sold every day for two years. Sales dip on weekends, climb slowly as the campus grows, and a slow day tends to be followed by another. Here you cut the series into windows of past values, split them in time order, score two baselines that need no training, fit a linear autoregression by least squares, and forecast the week after the data ends.

**How to work through it.** Each part asks for one or two functions. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your code is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first. It builds the lesson's twelve-day series and two years of daily sales with a seeded random generator.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)

# The lesson's twelve-day series. Every number in its checks is exact.
y_small = np.array([4, 6, 5, 7, 6, 8, 7, 9, 8, 10, 9, 11], dtype=float)

# Two years of daily coffee sales at a campus cafe, in cups. Day 0 is a Monday.
# Weekends are slow, the level climbs by about 36 cups over the two years, and
# the noise carries over from one day to the next.
rng = np.random.default_rng(0)
n_days = 730
day = np.arange(n_days)
weekly = np.array([10, 15, 20, 20, 25, -40, -50])[day % 7]        # Monday to Sunday
noise = np.zeros(n_days)
for i in range(1, n_days):
    noise[i] = 0.6 * noise[i - 1] + rng.normal(0, 8)
sales = np.round(200 + 0.05 * day + weekly + noise)
print("days:", n_days, "  first two weeks:", sales[:14])

plt.figure(figsize=(9, 3))
plt.plot(day, sales, lw=0.8)
plt.xlabel("day")
plt.ylabel("cups")
plt.title("two years of daily sales")
plt.show()

## Part 1 (coding task): Cut the series into windows

A model wants rows, so the series is cut into windows. With lookback $L$ and horizon $H$, each window takes $L$ consecutive values as its input and the value $H$ steps after the last of them as its target, so a series of length $n$ gives $n - L - H + 1$ windows. Write `make_windows(y, L, H)`, which returns `(X, target, t)`: the inputs as a matrix with one window per row, the targets, and the day of each target. The columns of `X` are lag features. With $H = 1$ the last column is the value one day before the target and the first is the value $L$ days before it.

A loop over the start of each window is fine. The lesson's series with $L = 3$ and $H = 1$ gives 9 windows, the first with input $(4, 6, 5)$ and target 7 on day 3.

In [ ]:
def make_windows(y, L, H):
    """(X, target, t): L past values per row, the value H steps after them, and that value's day."""
    # TODO
    raise NotImplementedError

In [ ]:
X_small, target_small, t_small = make_windows(y_small, 3, 1)
print("first window:", X_small[0], "-> target", target_small[0], "on day", t_small[0])
assert X_small.shape == (9, 3), f"twelve days with L = 3 and H = 1 give 9 windows of 3, got {X_small.shape}"
assert np.array_equal(X_small[0], [4, 6, 5]) and target_small[0] == 7 and t_small[0] == 3, "the first window is (4, 6, 5) -> 7 on day 3"
assert np.array_equal(target_small, y_small[3:]) and np.array_equal(t_small, np.arange(3, 12)), "the targets are days 3 to 11"
X2, target2, t2 = make_windows(np.arange(20.0), 4, 3)
assert X2.shape == (14, 4) and np.array_equal(X2[0], [0, 1, 2, 3]) and target2[0] == 6, "with L = 4 and H = 3 the first window (0, 1, 2, 3) has target 6"
assert np.array_equal(target2, t2), "on a series whose values are their own days, every target equals its day"

L = 14
X, target, t = make_windows(sales, L, 1)
assert X.shape == (716, 14), f"730 days with L = 14 give 716 windows, got {X.shape}"
assert np.array_equal(X[:, -1], sales[t - 1]) and np.array_equal(X[:, -7], sales[t - 7]), "the last column is lag 1, and the column 7 from the end is lag 7"
print("cafe windows:", X.shape)
print("Part 1 passed")

## Part 2 (coding task): Split in time order

The split between training and test is made by the day of the target. Write `split_by_target(t, cut)`, which returns `(train, test)`, two boolean masks over the windows: a window trains when its target comes before day `cut`, and tests otherwise. A window whose input ends before the cut can still have its target after it, and the lesson counts what the wrong rule, splitting by where the input ends, lets through: on the twelve-day series with the cut at day 9 it trains on a seventh window whose target, 10, is a validation value.

The cafe's test period is its last 90 days.

In [ ]:
def split_by_target(t, cut):
    """(train, test): masks for the windows whose target is before day `cut`, and the rest."""
    # TODO
    raise NotImplementedError

In [ ]:
train_small, test_small = split_by_target(t_small, 9)
assert train_small.dtype == bool and test_small.dtype == bool, "return boolean masks"
assert train_small.sum() == 6 and test_small.sum() == 3, "targets on days 3 to 8 train, and days 9 to 11 test"
assert np.array_equal(target_small[test_small], [10, 9, 11]), "the validation targets are 10, 9, 11"
assert not np.any(train_small & test_small) and np.all(train_small | test_small), "every window is in exactly one set"
by_input = (t_small - 1) < 9                                 # the wrong rule: the input ends before the cut
print("training windows by target:", train_small.sum(), "  by input end:", by_input.sum(), "  leaked target:", target_small[by_input & ~train_small])
assert by_input.sum() == 7, "the input-end rule lets one validation target into training"

cut = n_days - 90
train, test = split_by_target(t, cut)
print(f"cafe: {train.sum()} training windows, {test.sum()} test windows, test targets on days {t[test].min()} to {t[test].max()}")
assert test.sum() == 90 and t[train].max() == cut - 1, "the last 90 days are the test period"
print("Part 2 passed")

## Part 3 (coding task): Two forecasts that need no training

Write `naive(X)`, which forecasts the last value of each window, and `seasonal_naive(X, m)`, which forecasts the value one season of length `m` before the target. With $H = 1$ that is the column `m` from the end of each window, and it needs $L \geq m$. For the lesson's series, with a season of 2 days, the naive forecasts for the three validation days are $8, 10, 9$ and the seasonal ones $9, 8, 10$. For the cafe the season is 7 days, so seasonal naive predicts the same weekday last week.

In [ ]:
def naive(X):
    """The last value of each window."""
    # TODO
    raise NotImplementedError


def seasonal_naive(X, m):
    """The value m steps before each window's target: the column m from the end."""
    # TODO
    raise NotImplementedError

In [ ]:
assert np.array_equal(naive(X_small[test_small]), [8, 10, 9]), "naive repeats the last value: 8, 10, 9"
assert np.array_equal(seasonal_naive(X_small[test_small], 2), [9, 8, 10]), "seasonal naive with m = 2 gives 9, 8, 10"
assert np.array_equal(seasonal_naive(X_small, 1), naive(X_small)), "a season of 1 is the naive forecast"

f_naive, f_seasonal = naive(X[test]), seasonal_naive(X[test], 7)
assert f_naive.shape == (90,) and f_seasonal.shape == (90,), "one forecast per test window"
assert np.array_equal(f_seasonal, sales[t[test] - 7]), "seasonal naive on the cafe is the same weekday last week"

plt.figure(figsize=(9, 3))
plt.plot(t[test][:28], target[test][:28], "k.-", label="actual")
plt.plot(t[test][:28], f_naive[:28], label="naive")
plt.plot(t[test][:28], f_seasonal[:28], label="seasonal naive")
plt.xlabel("day")
plt.ylabel("cups")
plt.title("the first four test weeks")
plt.legend()
plt.show()
print("Part 3 passed")

## Part 4 (coding task): Score the forecasts

Write `mae(forecast, actual)`, the mean absolute error, in cups, and `mape(forecast, actual)`, the mean absolute percentage error: the absolute error divided by the actual value, averaged, times 100. Both return a float. MAPE compares series of different sizes, and it blows up when an actual value is near zero, which the cafe never has.

On the lesson's validation days the naive forecast has MAE $5/3$ and seasonal naive has MAE 1, and their MAPEs are 16.43% and 10.07%.

In [ ]:
def mae(forecast, actual):
    """Mean absolute error, in the series' own units."""
    # TODO
    raise NotImplementedError


def mape(forecast, actual):
    """Mean absolute percentage error."""
    # TODO
    raise NotImplementedError

In [ ]:
truth_small = target_small[test_small]
assert np.isclose(mae(naive(X_small[test_small]), truth_small), 5 / 3), "naive MAE on the lesson's series is 5/3"
assert np.isclose(mae(seasonal_naive(X_small[test_small], 2), truth_small), 1.0), "seasonal naive MAE is 1"
assert np.isclose(mape(naive(X_small[test_small]), truth_small), 16.43, atol=0.01), "naive MAPE is 16.43%"
assert np.isclose(mape(seasonal_naive(X_small[test_small], 2), truth_small), 10.07, atol=0.01), "seasonal naive MAPE is 10.07%"
assert mae(np.array([1.0, 2.0]), np.array([1.0, 2.0])) == 0, "a perfect forecast scores 0"

truth = target[test]
scores = {"naive": (mae(f_naive, truth), mape(f_naive, truth)),
          "seasonal naive": (mae(f_seasonal, truth), mape(f_seasonal, truth))}
for name, (m, p) in scores.items():
    print(f"{name:<16} MAE {m:6.2f} cups   MAPE {p:5.2f}%")
assert scores["seasonal naive"][0] < scores["naive"][0], "the same weekday last week should beat yesterday on a series with a weekly pattern"
print("Part 4 passed")

## Part 5 (coding task): A linear autoregression

An autoregression predicts the next value as a weighted sum of the last $L$ values plus a constant, and least squares picks the weights that fit the training windows best. Write `fit_ar(X, target)`, which returns `(w, b)`: the weights, one per lag, and the intercept as a float. Add a column of ones to `X` with `np.column_stack` and hand the system to `np.linalg.lstsq(A, target, rcond=None)`, whose first return value is the solution, with the intercept as its last entry. Then write `predict_ar(w, b, X)`, which is `X @ w + b`.

Fit it on the training windows only and score it on the test period. A model has to beat both baselines before it has learned anything the season did not already give it.

In [ ]:
def fit_ar(X, target):
    """(w, b): least-squares weights for the lags and the intercept."""
    # TODO
    raise NotImplementedError


def predict_ar(w, b, X):
    """The autoregression's forecast for each window."""
    # TODO
    raise NotImplementedError

In [ ]:
rng_check = np.random.default_rng(1)
X_check = rng_check.normal(size=(50, 3))
w_check, b_check = fit_ar(X_check, X_check @ np.array([0.5, -0.2, 0.3]) + 2.0)
assert np.allclose(w_check, [0.5, -0.2, 0.3]) and np.isclose(b_check, 2.0), "on data that is exactly linear in its lags the fit recovers the weights and the intercept"
assert np.allclose(predict_ar(w_check, b_check, X_check[:2]), X_check[:2] @ w_check + b_check), "predict_ar is X @ w + b"

w, b = fit_ar(X[train], target[train])
assert w.shape == (14,), f"one weight per lag, got shape {w.shape}"
f_ar = predict_ar(w, b, X[test])
scores["autoregression"] = (mae(f_ar, truth), mape(f_ar, truth))
for name, (m, p) in scores.items():
    print(f"{name:<16} MAE {m:6.2f} cups   MAPE {p:5.2f}%")
print("weights on lags 14 to 1:", w)
assert scores["autoregression"][0] < scores["seasonal naive"][0], "the autoregression should beat the seasonal naive baseline on the test period"

plt.figure(figsize=(9, 3))
plt.plot(t[test], truth, "k.-", lw=0.8, label="actual")
plt.plot(t[test], f_seasonal, lw=1, label="seasonal naive")
plt.plot(t[test], f_ar, lw=1.5, label="autoregression")
plt.xlabel("day")
plt.ylabel("cups")
plt.title("the 90 test days")
plt.legend()
plt.show()
print("Part 5 passed")

## Part 6 (coding task): The week after the data ends

The model predicts one day ahead. To go further, feed each prediction back in as the newest value and predict again, which is recursive forecasting. Write `recursive_forecast(w, b, window, steps)`, which starts from the last $L$ values of a series and returns the next `steps` forecasts as an array. Each step drops the oldest value of the window and appends the newest forecast.

The check runs it three times: first on toy weights that should give a flat line, then from the end of the training period into the first test week, where the actual values are known, and from the last window of the data into the week nobody has seen. Day 730 is a Wednesday, so the forecast should dip on its fourth and fifth days, the weekend.

In [ ]:
def recursive_forecast(w, b, window, steps):
    """`steps` forecasts from the last L values, each fed back in as the newest value."""
    # TODO
    raise NotImplementedError

In [ ]:
repeat_last = np.zeros(14)
repeat_last[-1] = 1.0
assert np.allclose(recursive_forecast(repeat_last, 0.0, sales[:14], 5), sales[13]), "weights that copy the last value forecast a flat line"

first_week = recursive_forecast(w, b, sales[cut - L:cut], 7)
assert first_week.shape == (7,), f"seven forecasts, got shape {first_week.shape}"
assert np.isclose(first_week[0], predict_ar(w, b, sales[cut - L:cut][None, :])[0]), "the first step is the one-step forecast"
actual_week = sales[cut:cut + 7]
print(f"first test week, seven days ahead: recursive MAE {mae(first_week, actual_week):.2f} cups, seasonal naive MAE {mae(sales[cut - 7:cut], actual_week):.2f}")

beyond = recursive_forecast(w, b, sales[-L:], 7)
print("the week after the data, Wednesday to Tuesday:", beyond)
assert beyond[3:5].max() < beyond[[0, 1, 2, 5, 6]].min(), "the forecast should carry the weekend dip"

plt.figure(figsize=(9, 3))
plt.plot(day[-42:], sales[-42:], "k.-", lw=0.8, label="last six weeks")
plt.plot(n_days + np.arange(7), beyond, "o-", label="next week, recursive")
plt.xlabel("day")
plt.ylabel("cups")
plt.legend()
plt.show()
print("Part 6 passed")

## Solution

### Part 1: Cut the series into windows

In [ ]:
def make_windows(y, L, H):
    """(X, target, t): L past values per row, the value H steps after them, and that value's day."""
    starts = np.arange(len(y) - L - H + 1)
    X = np.array([y[s:s + L] for s in starts])
    t = starts + L + H - 1
    return X, y[t], t

### Part 2: Split in time order

In [ ]:
def split_by_target(t, cut):
    """(train, test): masks for the windows whose target is before day `cut`, and the rest."""
    train = t < cut
    return train, ~train

### Part 3: Two forecasts that need no training

In [ ]:
def naive(X):
    """The last value of each window."""
    return X[:, -1]


def seasonal_naive(X, m):
    """The value m steps before each window's target: the column m from the end."""
    return X[:, -m]

### Part 4: Score the forecasts

In [ ]:
def mae(forecast, actual):
    """Mean absolute error, in the series' own units."""
    return float(np.mean(np.abs(forecast - actual)))


def mape(forecast, actual):
    """Mean absolute percentage error."""
    return float(np.mean(np.abs(forecast - actual) / np.abs(actual)) * 100)

### Part 5: A linear autoregression

In [ ]:
def fit_ar(X, target):
    """(w, b): least-squares weights for the lags and the intercept."""
    A = np.column_stack([X, np.ones(len(X))])
    coef = np.linalg.lstsq(A, target, rcond=None)[0]
    return coef[:-1], float(coef[-1])


def predict_ar(w, b, X):
    """The autoregression's forecast for each window."""
    return X @ w + b

### Part 6: The week after the data ends

In [ ]:
def recursive_forecast(w, b, window, steps):
    """`steps` forecasts from the last L values, each fed back in as the newest value."""
    window = np.array(window, dtype=float)
    out = []
    for _ in range(steps):
        nxt = float(window @ w + b)
        out.append(nxt)
        window = np.append(window[1:], nxt)
    return np.array(out)